### Data Reading

In [0]:
df=spark.read.format('csv').option('inferSchema', True).option('header', True).load('/Volumes/workspace/default/raw_files/BigMart Sales.csv')
df.show()

In [0]:
df.display()

In [0]:
# Get the total Count

print("Total rows:", df.count())

In [0]:
df = spark.sql("""
    DESCRIBE EXTENDED workspace.default.big_mart_sales
""")

display(df)

In [0]:
files = dbutils.fs.ls("/Volumes/workspace/default/raw_files/")

for f in files:
    print(f.path, f.size)

### Data Reading JSON

In [0]:
df_json = spark.read.format('json').option('inferSchema', True)\
                    .option('header', True)\
                    .option('multiLine', False)\
                    .load('/Volumes/workspace/default/raw_files/drivers.json')

In [0]:
df_json.display()

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/raw_files/")

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
    Item_Identifier string,
    Item_Weight string,
    Item_Fat_Content string,
    Item_Visibility double,
    Item_Type string,
    Item_MRP double,
    Outlet_Identifier string,
    Outlet_Establishment_Year integer,
    Outlet_Size string,
    Outlet_Location_Type string,
    Outlet_Type string,
    Item_Outlet_Sales double
'''

In [0]:
df = spark.read.format('csv')\
    .schema(my_ddl_schema)\
    .option('header', True)\
    .load('/Volumes/workspace/default/raw_files/BigMart Sales.csv')


In [0]:
df.printSchema()

In [0]:
df.display()

### StructType() Schema

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 


In [0]:


my_struct_schema = StructType([
                        StructField('Item_Identifier', StringType(), True),
                        StructField('Item_Weight', StringType(), True),
                        StructField('Item_Fat_Content', StringType(), True),
                        StructField('Item_Visibility', StringType(), True),
                        StructField('Item_Type', StringType(), True),
                        StructField('Item_MRP', StringType(), True),
                        StructField('Outlet_Identifier', StringType(), True),
                        StructField('Outlet_Establishment', StringType(), True),
                        StructField('Outlet_Size', StringType(), True),
                        StructField('Outlet_Location_Type', StringType(), True),
                        StructField('Outlet_Type', StringType(), True),
                        StructField('Item_Outlet_Sales', StringType(),True)        
                                    
])

df=spark.read.format('csv')\
    .schema(my_struct_schema)\
    .option('header', True)\
    .load("/Volumes/workspace/default/raw_files/BigMart Sales.csv")
df.printSchema()
df.display()

### SELECT

In [0]:
df_sel = df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

In [0]:
df_select_col = df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

### ALIAS

In [0]:
# alias cannot be used without col object

df_select_alias = df.select(col('Item_Identifier').alias('Item_ID'))

display(df_select_alias)

### FILTER

#### Scenario - 1

In [0]:
df.filter(col("Item_Fat_Content") == "Regular").display()

### Scenario - 2

In [0]:
df.filter( (col('Item_Type') == "Soft Drinks") & (col('Item_Weight') < 10) ).display()

### Scenario - 3

In [0]:
df.filter( (col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1', 'Tier 2'))).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight', 'Item_wt').display()

### withColumn

#### Scenario - 1

In [0]:
from pyspark.sql.functions import lit

df = df.withColumn('flag', lit("new"))

df.display()

In [0]:
df.withColumn('multiply', col('Item_Weight') * col('Item_MRP')).display()

#### Scenario - 2

In [0]:
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), "Regular", "Reg"))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), "Low Fat", "LF")).display()

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(IntegerType()))

df.printSchema()

### SORTING

#### Scenario - 1

In [0]:
df.sort(col('Item_Weight').desc()).display()

#### Scenario - 2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

#### Scenario - 3

In [0]:
df.sort(['Item_Visibility', 'Item_Weight'],ascending=[0,0]).display()

#### Scenario - 4

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0,1]).display()

### LIMIT

In [0]:
df.limit(10).display()

### DROP

#### Scenario - 1

In [0]:
df.drop('Item_Visibility').display()

#### Scenario - 2

In [0]:
df.drop('Item_Visibility', 'Item_Type').display()

### DROP_DUPLICATES

#### Scenario - 1

In [0]:
df.dropDuplicates().display()

#### Scenario - 2


In [0]:
df.dropDuplicates(subset = ['Item_Type']).display()

### UNION & UNION_BYNAME

In [0]:
data1 = [('1', 'kad'), ('2', 'sid')]
schema1 = "id STRING, name STRING"

data2 = [('3', 'Rahul'), ('4', 'Jas')]
schema2 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

# df1.display()
# df2.display()

df1.union(df2).display()



### UNION BY NAME


In [0]:
data1 = [('kad', '1'), ('2', 'sid')]
schema1 = "name STRING, id STRING"

data2 = [('3', 'Rahul'), ('4', 'Jas')]
schema2 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1)
df2 = spark.createDataFrame(data2, schema2)

df1.unionByName(df2).display()

### STRING FUNCTIONS 
##### INITCAP()
##### UPPER()
##### LOWER()

In [0]:
df.select(initcap('Item_Type')).limit(2).display()
df.select(lower('Item_Type')).limit(2).display()
df.select(upper('Item_Type')).limit(2).display()
df.select(upper('Item_Type').alias('Upper_Item_Type')).limit(2).display()

### DATE FUNCTIONS 

#### CURRENT DATE

In [0]:
df = df.withColumn('current_date', current_date())
df.display()

#### date_add() Function

In [0]:
df = df.withColumn('week_after', date_add('current_date', 7))
df.display()

#### Date_Sub() Function

In [0]:
df = df.withColumn('week_before', date_sub('current_date', 7))
df.display()

In [0]:
df = df.withColumn('week_before', date_add('current_date', -7))
df.display()


### DateDIFF()


In [0]:
df = df.withColumn('date_diff', datediff('week_after', 'current_date'))
df.display()

### DATE_FORMAT()

In [0]:
# dd	Day of month	20
# MM	Month of year	10
# yyyy	Calendar year	2026
# YYYY	Week-based year	Depends on the week and calendar rules
# HH	Hour (24-hour clock)	14
# mm	Minute	35
# ss	Second	42
df = df.withColumn('week_before', date_format('week_before', 'dd-MM-yyyy'))
df.display()

### Handling NULLs

#### DRopping NULLs


In [0]:
df.dropna('all')display()

In [0]:
df.dropna('any').display()

In [0]:
#GOOD TO USE 
df.dropna(subset=['Outlet_Size']).display()

### Filling NULLs

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast('string'))

df=df.fillna("Not Availble").display()


In [0]:
df.fillna("Not Available", subset=['Outlet_Size']).display()

In [0]:
df.display()

### SPLIT & INDEXING

#### SPLIT

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')).display()

#### INDEXING

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')[1]).display()

### EXPLODE

In [0]:
df_exp = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type', explode('Outlet_Type')).display()

In [0]:
df_exp.withColumn('Type1_Flag', array_contains('Outlet_Type', 'Type1')).display()


### GROUP BY 
#### Scenario - 1

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

#### Scenario - 2

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP').alias('Avg_MRP')).display()

#### Scenario - 4

In [0]:
df.groupBy('Item_Type', 'Outlet_Size').agg(sum('Item_MRP').alias('Sum_of_MRP')).display()

In [0]:
df.groupBy('Item_Type', 'Outlet_Size').agg(sum('Item_MRP'), avg('Item_MRP')).display()

### COLLECT_LIST

In [0]:
data = [
    ('user1', 'book1'),
    ('user2', 'book2'),
    ('user2', 'book2'),
    ('user2', 'book4'),
    ('user3', 'book1')
]

schema = 'user string, book string'

df_book = spark.createDataFrame(data, schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

### PIVOT

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

###    WHEN - OTHERWISE

#### Scenario - 1

In [0]:
df = df.withColumn('veg_flag', when(col('Item_Type') == 'Meat', 'Non-Veg').otherwise('Eww Vegetarian'))

In [0]:
df.withColumn('veg_exp_flag', when( ( (col('veg_flag') == "Eww Vegetarian") & (col('Item_MRP') < 100) ), 'Veg_Inexpensive' )\
                             .when( ( (col('veg_flag') == 'Eww Vegetarian') & (col('Item_MRP') > 100) ), 'Veg_Expensive')\
                             .otherwise('Non-Vegetarian')
               ).display()

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

### INNER JOINS 

In [0]:
# if we dont mention the column name at the start, the spark will face the error of ambiguity
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'inner').display()

### LEFT JOIN

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'left').display()

### RIGHT JOIN

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'right').display()

### ANTI-JOIN

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'anti').display()

### WINDOW FUNCTIONS

In [0]:
from pyspark.sql.window import Window

#### ROW_NUMBER()

In [0]:
df.withColumn('rowCol', row_number().over(Window.orderBy('Item_Identifier'))).display()

### RANK VS DENSE_RANK

In [0]:
df.withColumn('rank', rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('dense_rank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

### CUMULATIVE SUM

In [0]:
df.withColumn('cumulative_sum', sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cum_sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

### USER DEFINED FUNCTIONS (UDF)

#### STEP - 1

In [0]:
def my_func(x):
    return x*x

#### STEP - 2

In [0]:
my_udf = udf(my_func)

#### STEP - 3

In [0]:
df.withColumn('mynewCol', my_udf('Item_MRP')).display()

### DATA WRITING

#### CSV

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
SHOW SCHEMAS;

In [0]:
%sql
SHOW VOLUMES IN workspace.default;


In [0]:
df.write \
    .format('csv') \
    .option('header', 'true') \
    .mode('overwrite') \
    .save('/Volumes/workspace/default/raw_files/data.csv')

#### APPEND()

In [0]:
df.write.format('csv')\
    .mode('append')\
    .save('/Volumes/workspace/default/raw_files/data.csv')

In [0]:
df.write.format('csv')\
    .mode('append')\
    .option('path','/Volumes/workspace/default/raw_files/data.csv')\
    .save()

#### OVERWRITE()

In [0]:
df.write.format('csv')\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/default/raw_files/data.csv')\
    .save()

#### ERROR()

In [0]:
df.write.format('csv')\
    .mode('error')\
    .option('path', '/Volumes/workspace/default/raw_files/data.csv')\
    .save()

#### IGNORE()

In [0]:
df.write.format('csv')\
    .mode('ignore')\
    .option('path', '/Volumes/workspace/default/raw_files/data.csv')\
    .save()

#### PARQUET

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/default/raw_files/data.csv')\
    .save()

#### TABLE

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .saveAsTable('my_table')

### SPARK SQL

#### Creating Temp View

In [0]:
df.createTempView('my_view') 

In [0]:
%sql
SELECT * FROM my_view;

In [0]:
%sql
SELECT *
FROM INFORMATION_SCHEMA.VIEWS;

In [0]:
%sql
SELECT
    TABLE_SCHEMA,
    TABLE_NAME
FROM INFORMATION_SCHEMA.VIEWS
WHERE TABLE_SCHEMA = 'dbo';

In [0]:
df_sql = spark.sql('SELECT * FROM my_view;')
df_sql.display()